In [1]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
from optuna.samplers import TPESampler
import sys
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..'))) 
import script_otim as script
import logging

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "tpe_seed1" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "False"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_baa93807-7a19-433b-b214-4fea0d9f34ec.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 65314.
PyAEDT INFO: Electronics Desktop started on gRPC port: 65314 after 26.666749477386475 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 10344.


In [4]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    df_filtrado = df[(df['Frequency [MHz]'] >= 0.3) & (df['Frequency [MHz]'] <= 1.5)]
    
    s11 = np.asarray(df_filtrado['|$S_{11}$| [dB]'], dtype=float)

    proportion_bad = float(np.mean(s11 >= -10))

    return proportion_bad

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name,
    sampler=TPESampler(seed=123),
    load_if_exists=True
)

#seed1 = 123
#seed2 = 173
#seed3 = 247

[I 2026-03-16 10:08:17,278] Using an existing study with name 'tpe_seed1' instead of creating a new one.


Using an existing study with name 'tpe_seed1' instead of creating a new one.


In [5]:
study.optimize(objective, n_trials=1)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 10344!
PyAEDT INFO: Project Project561 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_ZBH' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Union of 2 objects has been executed.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed El

[I 2026-03-16 10:34:47,126] Trial 462 finished with value: 0.04132231404958678 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 8.358491169201171, 'n3': 1.1350072821196924, 'n4': 45, 'perc': 0.6069953472296372, 'r': 97}. Best is trial 146 with value: 0.04132231404958678.


Trial 462 finished with value: 0.04132231404958678 and parameters: {'a1': 1.0, 'a2': 1.0, 'n1': 33, 'n2': 8.358491169201171, 'n3': 1.1350072821196924, 'n4': 45, 'perc': 0.6069953472296372, 'r': 97}. Best is trial 146 with value: 0.04132231404958678.


In [6]:
len(study.trials)

463

In [7]:
study.best_params

{'a1': 1.0,
 'a2': 1.0,
 'n1': 92,
 'n2': 8.050074122636886,
 'n3': 1.5360630593944198,
 'n4': 38,
 'perc': 0.6134703421928958,
 'r': 107}

In [8]:
study.best_trial

FrozenTrial(number=146, state=TrialState.COMPLETE, values=[0.04132231404958678], datetime_start=datetime.datetime(2025, 12, 14, 23, 34, 30, 5845), datetime_complete=datetime.datetime(2025, 12, 14, 23, 59, 23, 38371), params={'a1': 1.0, 'a2': 1.0, 'n1': 92, 'n2': 8.050074122636886, 'n3': 1.5360630593944198, 'n4': 38, 'perc': 0.6134703421928958, 'r': 107}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'a1': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'a2': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'n1': IntDistribution(high=100, log=False, low=1, step=1), 'n2': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n3': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n4': IntDistribution(high=100, log=False, low=1, step=1), 'perc': FloatDistribution(high=0.8, log=False, low=0.2, step=None), 'r': IntDistribution(high=150, log=False, low=90, step=1)}, trial_id=147, value=None)

In [9]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True